In [1]:
import os
import ast
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 1000)

from tqdm import tqdm

from openai import AzureOpenAI

pd.set_option('display.max_columns', 100)

### Load OpenAI Model

In [2]:
os.environ["AZURE_OPENAI_KEY"] = ""
os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["AZURE_API_VERSION"] = ""
os.environ["AZURE_DEPLOYMENT_ID"] = ""
os.environ["AWS_ACCESS_KEY"] = ""
os.environ["AWS_SECRET_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"] = ""
model_name = "gpt-4o-mini"

client = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_deployment=os.getenv("AZURE_DEPLOYMENT_ID")
)

### Taxonomy Functions

In [3]:
def clean_and_parse(x):
    import ast 

    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        cleaned = x.replace('""', '"')
        cleaned = cleaned.replace('\n', ' ').replace('\r', '')  # Remove line breaks, if any

        try:
            return ast.literal_eval(cleaned)
        except Exception as e:
            print(f"Error parsing string: {cleaned}\n{e}")
            return x
    else:
        return x

def get_main_taxonomy_examples(dom: str, mode: str) -> str:

    if mode == "CC":
    
        df = pd.read_csv("../semeval-task-10/cc_taxonomy.csv")
        df = df.astype(str)
        df['Main Narrative Example'] = df['Main Narrative Example'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') and x.endswith(']') else x)
                
        taxonomy = ''
        examples = ''

        dom = dom.split(":")[0] if len(dom.split(":")) > 0 else dom

        if dom in df['Main Narrative'].values:
            temp = df[df['Main Narrative'] == dom].reset_index()         
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Main Narrative']} | Definition: {temp.loc[0, 'Main Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Main Narrative Example'], list):
                for item in temp.loc[0, 'Main Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Main Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Main Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Main Narrative']}\n"
            else:
                pass


        return (taxonomy, examples)

    else:

        df = pd.read_csv("../semeval-task-10/urw_taxonomy.csv")
        df = df.astype(str)
        df['Main Narrative Example'] = df['Main Narrative Example'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') and x.endswith(']') else x)
          
        taxonomy = ''
        examples = ''

        dom = dom.split(":")[0] if len(dom.split(":")) > 0 else dom

        if dom in df['Main Narrative'].values:
            temp = df[df['Main Narrative'] == dom].reset_index()           
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Main Narrative']} | Definition: {temp.loc[0, 'Main Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Main Narrative Example'], list):
                for item in temp.loc[0, 'Main Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Main Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Main Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Main Narrative']}\n"
            else:
                pass
            
        return (taxonomy, examples)

def get_sub_taxonomy_examples(sub: str, mode: str) -> str:

    if mode == "CC":
        df = pd.read_csv("../semeval-task-10/cc_taxonomy.csv")
        df = df.astype(str)
        df['Sub Narrative Example'] = df['Sub Narrative Example'].apply(clean_and_parse)

        taxonomy = ''
        examples = ''

        sub = sub.split(":")[2][1:] if len(sub.split(":")) > 1 else sub

        if sub in df['Sub Narrative'].values:
            temp = df[df['Sub Narrative'] == sub].reset_index()
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Sub Narrative']} | Definition: {temp.loc[0, 'Sub Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Sub Narrative Example'], list):
                for item in temp.loc[0, 'Sub Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Sub Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Sub Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Sub Narrative']}\n"
            else:
                pass
        return (taxonomy, examples)

    else:
        df = pd.read_csv("../semeval-task-10/urw_taxonomy.csv")
        df = df.astype(str)
        df['Sub Narrative Example'] = df['Sub Narrative Example'].apply(clean_and_parse)

        taxonomy = ''
        examples = ''

        sub = sub.split(":")[2][1:] if len(sub.split(":")) > 1 else sub

        if sub in df['Sub Narrative'].values:
            temp = df[df['Sub Narrative'] == sub].reset_index()
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Sub Narrative']} | Definition: {temp.loc[0, 'Sub Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Sub Narrative Example'], list):
                for item in temp.loc[0, 'Sub Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Sub Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Sub Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Sub Narrative']}\n"
            else:
                pass
        return (taxonomy, examples)

In [4]:
def create_prompt(text, dom, sub, mode):
    # Helper function replacing quotation marks in the text:
    replace_qm = lambda s: s.replace('"', "'")

    if mode == "CC":
        # Update predicted_labels by slicing from the 4th character
        main_taxonomy, main_examples = get_main_taxonomy_examples(dom, mode)
        sub_taxonomy, sub_examples = get_sub_taxonomy_examples(sub, mode)
        context = f"""You will be given an article along with the dominant narrative and sub narrative associated with the article. 
        
        GOAL: Justify the choice of dominant and sub narratives assigned to the article. Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.

            INSTRUCTIONS:
            Read the provided text carefully.
            You will be given: 
            1.Taxonomy - definition of the narrative
            2.List of relevant examples - sentences which determine which justify the narrative and align with its defintion 
            3.Any additional idenyifying information for the narratives.
            Based on the given information give an explanation as to why the dominant and sub narratives are the correct choice for the text file.
            
            Note: Keep the output concise and to the point - ideally find relevant textual examples.

            DOMINANT NARRATIVE: {dom[4:]}
            TAXONOMY: {main_taxonomy}
            RELEVANT EXAMPLES: 
            {main_examples}

            SUB NARRATIVE: {sub[4:]}
            TAXONOMY: {sub_taxonomy}
            RELEVANT EXAMPLES:
            {sub_examples}
        """

        prompt = f'''{context}
        -------------------------------------------------------
        Based on the given Instructions, Taxonomies and Examples: Justify the choice of dominant and sub narratives assigned to the Climate Change article. 

        ARTICLE TEXT TO PREDICT: "{replace_qm(text)}" => '''
        
        return {
            "role": "user",
            "content": prompt
        }
        
    else:
        # Update predicted_labels by slicing from the 5th character
        main_taxonomy, main_examples = get_main_taxonomy_examples(dom, mode)
        sub_taxonomy, sub_examples = get_sub_taxonomy_examples(sub, mode)

        context = f"""You will be given an article along with the dominant narrative and sub narrative associated with the article. 
        
        GOAL: Justify the choice of dominant and sub narratives assigned to the article. Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.

            INSTRUCTIONS:
            Read the provided text carefully.
            You will be given: 
            1.Taxonomy - definition of the narrative
            2.List of relevant examples - sentences which determine which justify the narrative and align with its defintion 
            3.Any additional idenyifying information for the narratives.
            Based on the given information give an explanation as to why the dominant and sub narratives are the correct choice for the text file.
            
            Note: Keep the output concise and to the point - ideally find relevant textual examples.

            DOMINANT NARRATIVE: {dom[5:]}
            TAXONOMY: {main_taxonomy}
            RELEVANT EXAMPLES: 
            {main_examples}

            SUB NARRATIVE: {sub[5:]}
            TAXONOMY: {sub_taxonomy}
            RELEVANT EXAMPLES:
            {sub_examples}
        """

        prompt = f'''{context}
        -------------------------------------------------------
        Based on the given Instructions, Taxonomies and Examples: Justify the choice of dominant and sub narratives assigned to the Ukraine Russia War article. 

        ARTICLE TEXT TO PREDICT: "{replace_qm(text)}" => '''
        
        return {
            "role": "user",
            "content": prompt
        }

In [5]:
def get_embedded_json(embedded_str):
    import re
    try:
        # Extract only the list content using regex
        match = re.search(r"\[.*\]", embedded_str)
        if not match:
            return []  # Return empty list if no valid list is found
        
        cleaned_str = match.group(0)  # Extract the matched list portion

        # Convert to Python list safely
        return ast.literal_eval(cleaned_str)
    
    except (ValueError, SyntaxError):
        return []  # Return empty list if parsing fails

In [6]:
def generate_response(text:str, dom, sub, mode, temp):
    system_main = \
'''You are an expert trained to analyse and justify the choice of dominant and sub narratives assigned to a given article within 80 words.

Instructions:

Use ReACT (Reasoning and Contextual Text) to justify the choice of dominant and sub narratives assigned to the article.
Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.
Keep the output concise and to the point—ideally find relevant textual examples.

Categorization Rules:

Use the help of the provided taxonomy and examples to justify the choice of dominant and sub narratives assigned to the article.

OUTPUT FORMAT:
Return text in paragraph(s) format within 80 words.
'''

    
    prompt = create_prompt(text, dom, sub, mode)

    messages = [
        {"role": "system", "content": system_main},
        {"role": "user", "content": prompt.get("content", "")}
    ]
    
    response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=temp,
    max_tokens=100
    )   

    output = str(response.choices[0].message.content)
    return output


### Importing the data

In [7]:
df = pd.read_csv('SemEval 2025 Test Data/final_datasets/eng_gold_label_data.csv')

In [8]:
df.shape

(203, 5)

In [9]:
df.head()

,article_id,dominant_narrative,sub_narratives,ground_truth,text
0,EN_CC_100013.txt,CC: Criticism of climate movement,CC: Criticism of climate movement: Ad hominem ...,The text accuses climate activist Bill Gates f...,Bill Gates Says He Is ‘The Solution’ To Climat...
1,EN_CC_200321.txt,CC: Questioning the measurements and science,CC: Questioning the measurements and science: ...,There are inconsistencies in the predictions o...,New paper makes ‘increasing tropical cyclone f...
2,EN_CC_100005.txt,CC: Criticism of climate movement,none,The article talks about climate activists atta...,Climate Crazies Fail in Attempt to Vandalize A...
3,EN_UA_014637.txt,URW: Speculating war outcomes,URW: Speculating war outcomes: Russian army is...,The text conveys a narrative depicting negativ...,Putin’s masses of HIV-positive prisoners choos...
4,EN_UA_019640.txt,URW: Praise of Russia,URW: Praise of Russia: Russia has internationa...,Multiple paragraphs within the text present al...,"After North Korea’s Kim Jong Un, Putin and Xi ..."


In [10]:
df['mode'] = df['dominant_narrative'].apply(lambda x: 'CC' if x.split(':')[0] == 'CC' else 'URW')

In [11]:
df['mode'].value_counts()

mode
URW    108
CC      95
Name: count, dtype: int64

In [12]:
df.head()

,article_id,dominant_narrative,sub_narratives,ground_truth,text,mode
0,EN_CC_100013.txt,CC: Criticism of climate movement,CC: Criticism of climate movement: Ad hominem ...,The text accuses climate activist Bill Gates f...,Bill Gates Says He Is ‘The Solution’ To Climat...,CC
1,EN_CC_200321.txt,CC: Questioning the measurements and science,CC: Questioning the measurements and science: ...,There are inconsistencies in the predictions o...,New paper makes ‘increasing tropical cyclone f...,CC
2,EN_CC_100005.txt,CC: Criticism of climate movement,none,The article talks about climate activists atta...,Climate Crazies Fail in Attempt to Vandalize A...,CC
3,EN_UA_014637.txt,URW: Speculating war outcomes,URW: Speculating war outcomes: Russian army is...,The text conveys a narrative depicting negativ...,Putin’s masses of HIV-positive prisoners choos...,URW
4,EN_UA_019640.txt,URW: Praise of Russia,URW: Praise of Russia: Russia has internationa...,Multiple paragraphs within the text present al...,"After North Korea’s Kim Jong Un, Putin and Xi ...",URW


In [13]:
tqdm.pandas()

for i in np.arange(0.1, 1, 0.1):
    print("Iterating at temperature: ", i.round(2))
    temp = i.round(2)
    df[f'output_temp_{temp}'] = df.progress_apply(lambda x: generate_response(x['text'], x['dominant_narrative'], x['sub_narratives'], x['mode'], temp), axis=1)

Iterating at temperature:  0.1


  0%|          | 0/203 [00:00<?, ?it/s]

100%|██████████| 203/203 [08:09<00:00,  2.41s/it]


Iterating at temperature:  0.2


100%|██████████| 203/203 [07:35<00:00,  2.24s/it]


...

100%|██████████| 203/203 [07:28<00:00,  2.21s/it]


Iterating at temperature:  0.8


100%|██████████| 203/203 [06:50<00:00,  2.02s/it]


Iterating at temperature:  0.9


100%|██████████| 203/203 [06:38<00:00,  1.96s/it]


In [14]:
df.to_csv('SemEval 2025 Test Data/final_datasets/pred_eng_gold_label_data.csv', index=False)

In [15]:
df.head()

,article_id,dominant_narrative,sub_narratives,ground_truth,text,mode,output_temp_0.1,output_temp_0.2,output_temp_0.3,output_temp_0.4,output_temp_0.5,output_temp_0.6,output_temp_0.7,output_temp_0.8,output_temp_0.9
0,EN_CC_100013.txt,CC: Criticism of climate movement,CC: Criticism of climate movement: Ad hominem ...,The text accuses climate activist Bill Gates f...,Bill Gates Says He Is ‘The Solution’ To Climat...,CC,"The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative ""Criticism of climate m...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat..."
1,EN_CC_200321.txt,CC: Questioning the measurements and science,CC: Questioning the measurements and science: ...,There are inconsistencies in the predictions o...,New paper makes ‘increasing tropical cyclone f...,CC,"The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative, ""Questioning the measu...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea..."
2,EN_CC_100005.txt,CC: Criticism of climate movement,none,The article talks about climate activists atta...,Climate Crazies Fail in Attempt to Vandalize A...,CC,"The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative, ""Criticism of climate ...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat..."
3,EN_UA_014637.txt,URW: Speculating war outcomes,URW: Speculating war outcomes: Russian army is...,The text conveys a narrative depicting negativ...,Putin’s masses of HIV-positive prisoners choos...,URW,"The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out..."
4,EN_UA_019640.txt,URW: Praise of Russia,URW: Praise of Russia: Russia has internationa...,Multiple paragraphs within the text present al...,"After North Korea’s Kim Jong Un, Putin and Xi ...",URW,"The dominant narrative of ""Praise of Russia"" i...","The dominant narrative of ""Praise of Russia"" i...","The dominant narrative of ""Praise of Russia"" i...","The dominant narrative of ""Praise of Russia"" i...","The dominant narrative of ""Praise of Russia"" i...","The dominant narrative of ""Praise of Russia"" i...","The dominant narrative of ""Praise of Russia"" i...","The dominant narrative, ""Praise of Russia,"" is...","The dominant narrative ""Praise of Russia"" is j..."


In [16]:
df.to_excel('SemEval 2025 Test Data/final_datasets/pred_eng_gold_label_data.xlsx', index=False)

In [23]:
from huggingface_hub import login

login("")

from bert_score import score

for i in tqdm(np.arange(0.1, 1, 0.1)):
    temp = round(i, 2)
    predictions = df[f'output_temp_{temp}'].tolist()
    references = df['ground_truth'].tolist()

    P, R, F1 = score(predictions, references, lang="en")

    df[f'precision_{temp}'] = P.tolist()
    df[f'recall_{temp}'] = R.tolist()
    df[f'f1_{temp}'] = F1.tolist()

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /Users/rahulbouri/.cache/huggingface/token
Login successful


  0%|          | 0/9 [00:00<?, ?it/s]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 11%|█         | 1/9 [01:22<10:56, 82.03s/it]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 22%|██▏       | 2/9 [01:45<05:31, 47.36s/it]Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inferenc

In [27]:
df.head()

,article_id,dominant_narrative,sub_narratives,ground_truth,text,mode,output_temp_0.1,output_temp_0.2,output_temp_0.3,output_temp_0.4,output_temp_0.5,output_temp_0.6,output_temp_0.7,output_temp_0.8,output_temp_0.9,precision_0.1,recall_0.1,f1_0.1,precision_0.2,recall_0.2,f1_0.2,precision_0.3,recall_0.3,f1_0.3,precision_0.4,recall_0.4,f1_0.4,precision_0.5,recall_0.5,f1_0.5,precision_0.6,recall_0.6,f1_0.6,precision_0.7,recall_0.7,f1_0.7,precision_0.8,recall_0.8,f1_0.8,precision_0.9,recall_0.9,f1_0.9
0,EN_CC_100013.txt,CC: Criticism of climate movement,CC: Criticism of climate movement: Ad hominem ...,The text accuses climate activist Bill Gates f...,Bill Gates Says He Is ‘The Solution’ To Climat...,CC,"The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative ""Criticism of climate m...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...",0.833789,0.891836,0.861836,0.836507,0.886449,0.860754,0.844080,0.903586,0.872820,0.845863,0.896246,0.870326,0.836019,0.903352,0.868382,0.838933,0.897120,0.867052,0.848848,0.899781,0.873573,0.842755,0.898510,0.869740,0.845070,0.881263,0.862787
1,EN_CC_200321.txt,CC: Questioning the measurements and science,CC: Questioning the measurements and science: ...,There are inconsistencies in the predictions o...,New paper makes ‘increasing tropical cyclone f...,CC,"The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative, ""Questioning the measu...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...","The dominant narrative of ""Questioning the mea...",0.839221,0.901484,0.869239,0.841211,0.901577,0.870349,0.837726,0.895898,0.865836,0.842073,0.910106,0.874769,0.841293,0.906816,0.872826,0.839586,0.900686,0.869064,0.841958,0.916467,0.877634,0.841123,0.898608,0.868916,0.840108,0.907107,0.872323
2,EN_CC_100005.txt,CC: Criticism of climate movement,none,The article talks about climate activists atta...,Climate Crazies Fail in Attempt to Vandalize A...,CC,"The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative, ""Criticism of climate ...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...","The dominant narrative of ""Criticism of climat...",0.854390,0.865143,0.859733,0.850256,0.867002,0.858548,0.862825,0.885291,0.873913,0.854806,0.868699,0.861697,0.846786,0.869030,0.857764,0.850736,0.867242,0.858910,0.846967,0.878207,0.862304,0.844705,0.868347,0.856363,0.855721,0.879259,0.867331
3,EN_UA_014637.txt,URW: Speculating war outcomes,URW: Speculating war outcomes: Russian army is...,The text conveys a narrative depicting negativ...,Putin’s masses of HIV-positive prisoners choos...,URW,"The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...","The dominant narrative of ""Speculating war out...",0.862717,0.888160,0.875254,0.864836,0.885191,0.874895,0.859619,0.887270,0.8

In [25]:
df.to_csv('SemEval 2025 Test Data/final_datasets/pred_eng_gold_label_data.csv', index=False)

In [28]:
print(f"Mean Precision Temp 0.1: {df['precision_0.1'].mean()}")
print(f"Mean Recall Temp 0.1: {df['recall_0.1'].mean()}")
print(f"Mean F1 Temp 0.1: {df['f1_0.1'].mean()}")
print()
print(f"Mean Precision Temp 0.2: {df['precision_0.2'].mean()}")
print(f"Mean Recall Temp 0.2: {df['recall_0.2'].mean()}")
print(f"Mean F1 Temp 0.2: {df['f1_0.2'].mean()}")
print()
print(f"Mean Precision Temp 0.3: {df['precision_0.3'].mean()}")
print(f"Mean Recall Temp 0.3: {df['recall_0.3'].mean()}")
print(f"Mean F1 Temp 0.3: {df['f1_0.3'].mean()}")
print()
print(f"Mean Precision Temp 0.4: {df['precision_0.4'].mean()}")
print(f"Mean Recall Temp 0.4: {df['recall_0.4'].mean()}")
print(f"Mean F1 Temp 0.4: {df['f1_0.4'].mean()}")
print()
print(f"Mean Precision Temp 0.5: {df['precision_0.5'].mean()}")
print(f"Mean Recall Temp 0.5: {df['recall_0.5'].mean()}")
print(f"Mean F1 Temp 0.5: {df['f1_0.5'].mean()}")
print()
print(f"Mean Precision Temp 0.6: {df['precision_0.6'].mean()}")
print(f"Mean Recall Temp 0.6: {df['recall_0.6'].mean()}")
print(f"Mean F1 Temp 0.6: {df['f1_0.6'].mean()}")
print()
print(f"Mean Precision Temp 0.7: {df['precision_0.7'].mean()}")
print(f"Mean Recall Temp 0.7: {df['recall_0.7'].mean()}")
print(f"Mean F1 Temp 0.7: {df['f1_0.7'].mean()}")
print()
print(f"Mean Precision Temp 0.8: {df['precision_0.8'].mean()}")
print(f"Mean Recall Temp 0.8: {df['recall_0.8'].mean()}")
print(f"Mean F1 Temp 0.8: {df['f1_0.8'].mean()}")
print()
print(f"Mean Precision Temp 0.9: {df['precision_0.9'].mean()}")
print(f"Mean Recall Temp 0.9: {df['recall_0.9'].mean()}")
print(f"Mean F1 Temp 0.9: {df['f1_0.9'].mean()}")

Mean Precision Temp 0.1: 0.8413284829097428
Mean Recall Temp 0.1: 0.8752876725690119
Mean F1 Temp 0.1: 0.8579033542736411

Mean Precision Temp 0.2: 0.8419654275396188
Mean Recall Temp 0.2: 0.8754032700519844
Mean F1 Temp 0.2: 0.8582917537008014

Mean Precision Temp 0.3: 0.8415083885192871
Mean Recall Temp 0.3: 0.875730993418858
Mean F1 Temp 0.3: 0.8582142509263138

Mean Precision Temp 0.4: 0.8418233021726749
Mean Recall Temp 0.4: 0.8753126950686788
Mean F1 Temp 0.4: 0.8581766257145135

Mean Precision Temp 0.5: 0.8411582400058878
Mean Recall Temp 0.5: 0.8747857439106909
Mean F1 Temp 0.5: 0.8575637375779928

Mean Precision Temp 0.6: 0.8412248004833466
Mean Recall Temp 0.6: 0.8747263768036377
Mean F1 Temp 0.6: 0.857582064391357

Mean Precision Temp 0.7: 0.8408166865997126
Mean Recall Temp 0.7: 0.8737973491546556
Mean F1 Temp 0.7: 0.8569279318959843

Mean Precision Temp 0.8: 0.8409134663384537
Mean Recall Temp 0.8: 0.8746660219624712
Mean F1 Temp 0.8: 0.8573901964525871

Mean Precision Tem